# Детекция ботов-парсеров по событиям куки

Полное решение: `Run All` пересоздаёт `submission.csv` (~3 минуты). Описание подхода и результаты — в `README.md`,
исследования, на которых они основаны, — в `experiments.ipynb` (необязательно).

1. Загрузка данных
2. Очистка событий
3. Признаки: поведение куки, история и репутация объявлений
4. Модель
5. Валидация
6. Submission

## 1. Загрузка данных

In [1]:
import re
import sys

import catboost
import numpy as np
import pandas as pd
import sklearn
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score

from metric import precision_at_recall

SEED = 42
print('Python', sys.version.split()[0], '| pandas', pd.__version__, '| scikit-learn', sklearn.__version__,
      '| catboost', catboost.__version__)

META_DATES = ['cookie_created_at', 'window_start_ts', 'window_end_ts']
train = pd.read_csv('data/train.csv', parse_dates=META_DATES)
test = pd.read_csv('data/test.csv', parse_dates=META_DATES)
events_raw = pd.read_csv('data/events.csv.gz', parse_dates=['event_ts'])
meta = pd.concat([train.drop(columns='target'), test], ignore_index=True)
print('train', train.shape, '| test', test.shape, '| events', events_raw.shape)

Python 3.14.4 | pandas 3.0.6 | scikit-learn 1.9.1 | catboost 1.2.10


train (11091, 5) | test (4909, 4) | events (328905, 14)


## 2. Очистка событий

In [2]:
# 11 написаний platform → web / android / ios; desktop — это web-версия сайта
PLATFORM_MAP = {'web': 'web', 'desktop': 'web', 'android': 'android', 'ios': 'ios', 'iphone': 'ios'}
# User-Agent нельзя брать как есть: версии дробят один браузер на десятки категорий. Выделяем тип клиента,
# семейство браузера и major-версию. Библиотеки и утилиты представляются своим именем, а не браузером:
SCRIPT_UA = re.compile(r'curl|python|scrapy|go-http|node-fetch|wget|java/|httpclient', re.I)


def parse_user_agent(ua: str) -> dict:
    """Тип клиента, семейство браузера и major-версия из строки User-Agent."""
    if SCRIPT_UA.search(ua):
        return {'ua_client': 'script', 'ua_family': ua.split('/')[0].lower(), 'ua_major': np.nan}
    if ua.startswith('Avito/'):   # мобильное приложение: Avito/<версия> (Android|iPhone ...)
        return {'ua_client': 'app', 'ua_family': 'avito_app', 'ua_major': float(ua[6:].split('.')[0])}

    # порядок важен: YaBrowser и HeadlessChrome содержат подстроку Chrome, Chrome содержит Safari
    for family, pattern in [('yandex', r'YaBrowser/(\d+)'), ('headless_chrome', r'HeadlessChrome/(\d+)'),
                            ('firefox', r'Firefox/(\d+)'), ('chrome', r'Chrome/(\d+)'),
                            ('safari', r'Version/(\d+)')]:
        m = re.search(pattern, ua)
        if m:
            break
    else:
        family, m = 'other', None

    if family == 'headless_chrome':
        client = 'headless'
    elif 'Mobile' in ua:
        client = 'mobile_browser'
    else:
        client = 'desktop_browser'
    return {'ua_client': client, 'ua_family': family, 'ua_major': float(m.group(1)) if m else np.nan}


def user_agent_table(user_agents) -> pd.DataFrame:
    """Разобранный User-Agent для каждой уникальной строки."""
    return pd.DataFrame([{'user_agent': ua, **parse_user_agent(ua)} for ua in user_agents])


def prepare_events(events: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    """Очистка событий — одна функция для train и test, чтобы обработка гарантированно совпадала."""
    ev = events.drop(columns='eid').copy()                          # eid однозначно дублирует event_name
    ev['platform'] = ev.platform.str.lower().map(PLATFORM_MAP)
    assert ev.platform.notna().all(), 'встретилась неизвестная платформа'
    ev = ev.drop_duplicates()        # после нормализации: иначе дубль, отличающийся только регистром платформы, выживет

    # только события внутри окна [window_start_ts, window_end_ts): события после конца окна — это будущее (~12% строк)
    ev = ev.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id', validate='many_to_one')
    ev = ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)]

    ev = ev.merge(user_agent_table(events.user_agent.unique()), on='user_agent', how='left', validate='many_to_one')
    # файл не отсортирован; стабильная сортировка делает порядок событий с одинаковым временем воспроизводимым
    return ev.sort_values(['cookie_id', 'event_ts'], kind='mergesort').reset_index(drop=True)


events = prepare_events(events_raw, meta)
assert meta.cookie_id.isin(events.cookie_id).all(), 'у каждой куки должны быть события в окне'
print('событий: было', len(events_raw), '→ стало', len(events))

событий: было 328905 → стало 283830


## 3. Признаки
### 3.1. Поведение куки внутри её окна

In [3]:
# Одна строка на куку. Пропуски остаются NaN, если значение не определено (одно событие — нет интервала,
# мобильная кука — нет курсора, нет поиска — нет глубины выдачи): CatBoost обрабатывает NaN нативно.
SESSION_GAP = 30 * 60              # пауза, после которой начинается новая сессия, сек
SCREEN_W, SCREEN_H = 1920, 1080    # максимальные координаты курсора в данных
CATEGORICAL = ['platform', 'ua_client', 'ua_family']
# действия, которые требуют интереса к конкретному объявлению, а не просто просмотра страниц
ENGAGEMENT_EVENTS = ['photo_swipe', 'favorite_add', 'login',
                     'contact_phone_show', 'contact_chat_open', 'contact_message_sent']


def entropy(values: pd.Series) -> float:
    p = values.value_counts(normalize=True).to_numpy()
    return float(-(p * np.log2(p)).sum()) if len(p) else 0.0


def per_cookie_share(mask: pd.Series, ev: pd.DataFrame) -> pd.Series:
    return mask.groupby(ev.cookie_id).mean()


def timing_features(ev):
    """Ритм: интервалы между событиями, сессии, темп, суточная активность."""
    ev = ev.assign(dt=ev.groupby('cookie_id').event_ts.diff().dt.total_seconds())
    g = ev.groupby('cookie_id')
    n_gaps = (g.size() - 1).replace(0, np.nan)   # у куки с одним событием интервалов нет
    f = pd.DataFrame({
        'n_events': g.size(),
        'active_span_s': (g.event_ts.max() - g.event_ts.min()).dt.total_seconds(),
        'dt_min': g.dt.min(), 'dt_median': g.dt.median(), 'dt_mean': g.dt.mean(), 'dt_std': g.dt.std(),
        'dt_p10': g.dt.quantile(0.1), 'dt_p90': g.dt.quantile(0.9),
        'dt_share_le1': ev.dt.le(1).groupby(ev.cookie_id).sum() / n_gaps,
        'dt_share_le5': ev.dt.le(5).groupby(ev.cookie_id).sum() / n_gaps,
        'dt_share_ge300': ev.dt.ge(300).groupby(ev.cookie_id).sum() / n_gaps,
        # доля самого частого интервала: скрипт с фиксированной задержкой даёт один и тот же интервал
        'dt_top_value_share': g.dt.agg(lambda x: x.value_counts(normalize=True).iloc[0] if x.notna().any() else np.nan),
        'dt_entropy': g.dt.agg(lambda x: entropy(x.dropna().round())),
    })
    f['dt_cv'] = f.dt_std / f.dt_mean.replace(0, np.nan)

    # сессии: новая начинается после паузы длиннее SESSION_GAP
    session_id = (ev.dt.isna() | ev.dt.gt(SESSION_GAP)).groupby(ev.cookie_id).cumsum()
    s = ev.groupby(['cookie_id', session_id]).event_ts.agg(['size', 'min', 'max'])
    s['span'] = (s['max'] - s['min']).dt.total_seconds()
    sg = s.groupby('cookie_id')
    f['n_sessions'] = sg.size()
    f['session_events_max'] = sg['size'].max()
    f['session_span_max_s'] = sg.span.max()
    f['events_per_active_min'] = f.n_events / (sg.span.sum() / 60 + 1)   # +1 мин: сессия из одного события

    hour = ev.event_ts.dt.hour
    f['night_share'] = per_cookie_share(hour.between(0, 5), ev)
    f['n_hours'] = hour.groupby(ev.cookie_id).nunique()
    f['first_event_hour'] = g.event_ts.min().dt.hour
    return f

In [4]:
def content_features(ev):
    """Сценарий: что и как смотрит кука — разнообразие, глубина выдачи, доли типов событий, переходы."""
    g = ev.groupby('cookie_id')
    items = ev[ev.item_id.notna()].groupby('cookie_id').item_id
    search = ev[ev.event_name == 'search_results_view']
    page_step = search.groupby('cookie_id').search_page.diff()
    sg = search.groupby('cookie_id')
    has_seller = ev.seller_type.notna().groupby(ev.cookie_id).sum().replace(0, np.nan)

    f = pd.DataFrame({
        'item_nunique': g.item_id.nunique(),
        'item_repeat_share': 1 - items.nunique() / items.size(),
        'category_nunique': g.item_category.nunique(),
        'category_entropy': g.item_category.agg(entropy),
        'location_nunique': g.item_location.nunique(),
        'location_entropy': g.item_location.agg(entropy),
        'query_nunique': g.search_query.nunique(),
        'pro_seller_share': ev.seller_type.eq('pro').groupby(ev.cookie_id).sum() / has_seller,
        'search_page_max': sg.search_page.max(),
        'search_page_mean': sg.search_page.mean(),
        'search_page_gt1_share': search.search_page.gt(1).groupby(search.cookie_id).mean(),
        'search_next_page_share': page_step.eq(1).groupby(search.cookie_id).mean(),
        'searches_per_query': sg.size() / sg.search_query.nunique().replace(0, np.nan),
    })
    for name in sorted(ev.event_name.unique()):
        f[f'share_{name}'] = per_cookie_share(ev.event_name.eq(name), ev)
    f['n_captcha'] = ev.event_name.eq('captcha_shown').groupby(ev.cookie_id).sum()
    f['engagement_share'] = per_cookie_share(ev.event_name.isin(ENGAGEMENT_EVENTS), ev)
    f['photos_per_item_view'] = f.share_photo_swipe / f.share_item_view.replace(0, np.nan)
    f['items_per_search'] = f.share_item_view / f.share_search_results_view.replace(0, np.nan)

    # переходы «событие → следующее событие»: у человека сценарий разнообразнее
    next_event = g.event_name.shift(-1)
    transition = (ev.event_name + '>' + next_event).where(next_event.notna())
    f['n_transition_types'] = transition.groupby(ev.cookie_id).nunique()
    f['same_event_repeat_share'] = (ev.event_name.eq(next_event).groupby(ev.cookie_id).sum()
                                    / (g.size() - 1).replace(0, np.nan))
    return f

In [5]:
def pointer_features(ev):
    """Курсор на web: есть ли он, где он, разброс, клики точно по краю экрана."""
    # курсор пишется только для web; у мобильных кук эти признаки остаются NaN
    web = ev[ev.platform == 'web']
    p = web[web.pointer_x.notna()].copy()
    p['on_edge'] = p.pointer_x.isin([0, SCREEN_W]) | p.pointer_y.isin([0, SCREEN_H])
    p['round100'] = p.pointer_x.mod(100).eq(0) & p.pointer_y.mod(100).eq(0)
    pg = p.groupby('cookie_id')
    step = np.hypot(pg.pointer_x.diff(), pg.pointer_y.diff())   # смещение курсора между событиями
    return pd.DataFrame({
        'pointer_share': per_cookie_share(web.pointer_x.notna(), web),
        'pointer_x_mean': pg.pointer_x.mean(),
        'pointer_y_mean': pg.pointer_y.mean(),
        'pointer_x_std': pg.pointer_x.std(),
        'pointer_y_std': pg.pointer_y.std(),
        'pointer_edge_share': pg.on_edge.mean(),
        'pointer_round100_share': pg.round100.mean(),
        'pointer_step_median': step.groupby(p.cookie_id).median(),
        'pointer_unique_share': pg[['pointer_x', 'pointer_y']].apply(lambda d: len(d.drop_duplicates()) / len(d)),
    })


def client_features(ev):
    """Платформа и разобранный User-Agent."""
    g = ev.groupby('cookie_id')
    f = g[['platform', 'ua_client', 'ua_family', 'ua_major']].first()
    f['n_user_agents'] = g.user_agent.nunique()
    return f


def cookie_features(ev, meta):
    """Возраст куки."""
    m = meta.set_index('cookie_id')
    first_event = ev.groupby('cookie_id').event_ts.min()
    return pd.DataFrame({
        'cookie_age_days': (m.window_start_ts - m.cookie_created_at).dt.total_seconds() / 86400,
        'cookie_to_first_event_h': (first_event - m.cookie_created_at).dt.total_seconds() / 3600,
    })

### 3.2. История объявлений (history)

In [6]:
# Боты разных кук смотрят одни и те же объявления, поэтому популярные объявления подозрительны.
# Считается только по прошлым дням и без меток.
HISTORY_DAYS = 7


def values_by_day(ev, meta, column='item_id'):
    """Уникальные пары (кука, значение колонки) с днём окна куки."""
    views = ev[ev[column].notna()][['cookie_id', column]].drop_duplicates()
    views['day'] = views.cookie_id.map(meta.set_index('cookie_id').window_start_ts)
    return views


def history_features(ev, meta, lookback=HISTORY_DAYS):
    """Популярность объявлений куки за прошлые `lookback` дней: сколько кук их смотрели, на 1000 кук истории.

    Нормировка на число кук нужна потому, что в разные дни в выборку попадает от ~600 до ~900 кук:
    без неё масштаб признака дрейфует (в тестовую неделю кук в день меньше, чем в трейне).
    """
    views = values_by_day(ev, meta)
    cookies_per_day = meta.groupby('window_start_ts').size()
    parts = []
    for d in sorted(views.day.unique()):
        start = d - pd.Timedelta(days=lookback)
        history = views[(views.day >= start) & (views.day < d)]
        n_history = cookies_per_day[(cookies_per_day.index >= start) & (cookies_per_day.index < d)].sum()
        today = views[views.day == d]
        if n_history == 0:                     # первый день выборки: истории ещё нет
            parts.append(today.assign(pop=np.nan, seen=np.nan))
            continue
        pop = today.item_id.map(history.groupby('item_id').cookie_id.nunique()).fillna(0) / n_history * 1000
        parts.append(today.assign(pop=pop, seen=(pop > 0).astype(float)))
    g = pd.concat(parts).groupby('cookie_id')
    return pd.DataFrame({'hist_pop_mean': g['pop'].mean(), 'hist_pop_median': g['pop'].median(),
                         'hist_pop_max': g['pop'].max(), 'hist_seen_share': g['seen'].mean()})

### 3.3. Репутация объявлений (reputation)

In [7]:
# Какая доля кук, смотревших те же объявления в прошлые дни, оказалась ботами. Честность по времени: для куки дня d —
# только метки кук строго прошлых дней (< d); для test — только метки train; в недельных блоках валидации — только
# метки до начала блока. Доля ботов сглаживается к фиксированной константе REP_PRIOR с весом REP_WEIGHT просмотров:
# объявление с одним просмотром не получает репутацию 0 или 1, а одинаковая история даёт одинаковое значение в любой день.
REP_PRIOR, REP_WEIGHT = 0.1, 2.0


def reputation_features(ev, meta, known_labels, column='item_id', prefix='item'):
    """Репутация значений `column` у куки дня d по меткам `known_labels` кук строго прошлых дней (< d)."""
    views = values_by_day(ev, meta, column)
    views['t'] = views.cookie_id.map(known_labels)
    parts = []
    for d in sorted(views.day.unique()):
        history = views[(views.day < d) & views.t.notna()]
        today = views[views.day == d]
        if history.empty:                      # меток из прошлого ещё нет
            parts.append(today.assign(rep=np.nan, bot_seen=np.nan, evidence=np.nan))
            continue
        stats = history.groupby(column).t.agg(['sum', 'count'])
        rep = (stats['sum'] + REP_PRIOR * REP_WEIGHT) / (stats['count'] + REP_WEIGHT)
        rep_today = today[column].map(rep).fillna(REP_PRIOR)
        bot_seen = today[column].isin(stats.index[stats['sum'] > 0]).astype(float)
        # сумма свидетельств log(rep / prior) растёт с каждым подозрительным объявлением, а не усредняется
        parts.append(today.assign(rep=rep_today, bot_seen=bot_seen, evidence=np.log(rep_today / REP_PRIOR)))
    g = pd.concat(parts).groupby('cookie_id')
    return pd.DataFrame({f'{prefix}_rep_mean': g.rep.mean(), f'{prefix}_rep_max': g.rep.max(),
                         f'{prefix}_bot_seen_share': g.bot_seen.mean(),
                         f'{prefix}_bot_seen_count': g.bot_seen.sum(min_count=1),
                         f'{prefix}_rep_evidence': g.evidence.sum(min_count=1)})

### 3.4. Матрица признаков

In [8]:
def build_features(ev, meta):
    """Признаки без меток (индекс — cookie_id, порядок как в meta) и словарь группа → колонки."""
    parts = {'timing': timing_features(ev), 'content': content_features(ev), 'pointer': pointer_features(ev),
             'client': client_features(ev), 'cookie': cookie_features(ev, meta),
             'history': history_features(ev, meta)}
    X = pd.concat(parts.values(), axis=1).reindex(meta.cookie_id)
    for col in CATEGORICAL:
        X[col] = X[col].astype('category')
    return X, {name: list(part.columns) for name, part in parts.items()}


def with_reputation(X, labels_key='все метки'):
    return pd.concat([X, REPUTATION[labels_key].reindex(X.index)], axis=1)


X_all, FEATURE_GROUPS = build_features(events, meta)
X, X_test = X_all.loc[train.cookie_id], X_all.loc[test.cookie_id]

train_labels = train.set_index('cookie_id').target
label_day = train.set_index('cookie_id').window_start_ts
REPUTATION = {'все метки': reputation_features(events, meta, train_labels)}   # для финальной модели и test
FEATURE_GROUPS['reputation'] = list(REPUTATION['все метки'].columns)
X_full, X_test_full = with_reputation(X), with_reputation(X_test)
print('признаков:', X_full.shape[1], {g: len(c) for g, c in FEATURE_GROUPS.items()})

признаков: 74 {'timing': 21, 'content': 28, 'pointer': 9, 'client': 5, 'cookie': 2, 'history': 4, 'reputation': 5}


## 4. Модель

In [9]:
# CatBoost: симметричные деревья и упорядоченный бустинг меньше переобучаются на небольших данных, категории
# обрабатываются нативно. 1500 итераций с темпом 0.03, остальное по умолчанию; предсказание усредняется по трём seed.
CATBOOST_PARAMS = dict(iterations=1500, learning_rate=0.03, verbose=0, thread_count=-1, allow_writing_files=False)
CATBOOST_SEEDS = [42, 43, 44]


def as_catboost_input(X):
    return X.astype({c: str for c in CATEGORICAL})      # CatBoost принимает категории строками


def catboost_model(seed=SEED, **params):
    return CatBoostClassifier(**params, random_seed=seed, cat_features=CATEGORICAL)


def final_model(X_tr, y_tr, X_va):
    """Финальная модель: CatBoost, усреднённый по трём seed."""
    return np.mean([catboost_model(seed, **CATBOOST_PARAMS).fit(as_catboost_input(X_tr), y_tr)
                    .predict_proba(as_catboost_input(X_va))[:, 1] for seed in CATBOOST_SEEDS], axis=0)


def quickstart_rf(X_tr, y_tr, X_va):
    """Baseline из quickstart.ipynb: случайный лес на двух счётчиках."""
    cols = ['n_events', 'item_nunique']
    model = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=SEED, n_jobs=-1)
    return model.fit(X_tr[cols], y_tr).predict_proba(X_va[cols])[:, 1]


def constant(X_tr, y_tr, X_va):
    return np.full(len(X_va), 0.5)

## 5. Валидация

In [10]:
# Тест — неделя сразу после трейна, поэтому обе схемы временные:
# 1) однодневные фолды 12–19 апреля: модель учится на всех днях до фолда; метрики — на объединённых OOF-предсказаниях
#    (P@R0.7 из metric.py, сглаженный P@R по полноте 0.60…0.80, средний P@R0.7 по дням, PR-AUC);
# 2) недельные блоки, повторяющие тест: одна модель учится на днях до D и предсказывает следующие 7 дней,
#    а reputation считается только по меткам, известным до D (на тесте метки старше на 1–7 дней).
VALID_DAYS = pd.date_range('2026-04-12', '2026-04-19')
BLOCK_STARTS = [pd.Timestamp('2026-04-11'), pd.Timestamp('2026-04-13')]
RECALL_GRID = [0.60, 0.65, 0.70, 0.75, 0.80]
y = train.target.to_numpy()
day = train.window_start_ts

REPUTATION.update({s: reputation_features(events, meta, train_labels[label_day < s]) for s in BLOCK_STARTS})
X_full_blocks = {s: with_reputation(X, s) for s in BLOCK_STARTS}

results = {}        # сводная таблица экспериментов: имя → метрики
oof = {}            # out-of-fold предсказания по однодневным фолдам
block_scores = {}   # предсказания на недельных блоках


def block_masks(start):
    """Обучение — все дни до start, валидация — неделя [start, start + 7 дней)."""
    return (day < start).to_numpy(), ((day >= start) & (day < start + pd.Timedelta(days=7))).to_numpy()


def cv_predict(fit_predict, X, days=VALID_DAYS):
    """Out-of-fold score по однодневным фолдам. fit_predict(X_train, y_train, X_valid) -> score для X_valid."""
    score = pd.Series(np.nan, index=X.index)
    for d in days:
        tr, va = (day < d).to_numpy(), (day == d).to_numpy()
        score[va] = fit_predict(X[tr], y[tr], X[va])
    return score


def smoothed_par(y_true, score):
    return np.mean([precision_at_recall(y_true, score, r) for r in RECALL_GRID])


def metrics(score):
    m = score.notna().to_numpy()
    y_va, s_va = y[m], score[m].to_numpy()
    per_day = [precision_at_recall(y[(day == d).to_numpy()], score[(day == d).to_numpy()]) for d in VALID_DAYS]
    return {'P@R0.7': precision_at_recall(y_va, s_va), 'P@R сглаж.': smoothed_par(y_va, s_va),
            'P@R0.7 по дням': np.mean(per_day), 'PR-AUC': average_precision_score(y_va, s_va)}


def run_experiment(fit_predict, X, X_blocks=None):
    """Метрики на однодневных фолдах и недельных блоках (X_blocks — матрицы для блоков, если признаки зависят от меток)."""
    score = cv_predict(fit_predict, X)
    res, blocks = metrics(score), {}
    for start in BLOCK_STARTS:
        Xb = X if X_blocks is None else X_blocks[start]
        tr, va = block_masks(start)
        blocks[start] = fit_predict(Xb[tr], y[tr], Xb[va])
        res[f'неделя с {start:%d.%m}'] = precision_at_recall(y[va], blocks[start])
    return res, score, blocks


def evaluate(name, fit_predict, X, X_blocks=None):
    results[name], oof[name], block_scores[name] = run_experiment(fit_predict, X, X_blocks)
    return results[name]


def show_results(names=None):
    table = pd.DataFrame(results).T.astype(float).round(3)
    return table.loc[names] if names else table

In [11]:
FINAL = 'финальная модель: CatBoost, 3 seed'
evaluate('константа', constant, X)
evaluate('baseline из quickstart (RF на 2 признаках)', quickstart_rf, X)
evaluate(FINAL, final_model, X_full, X_full_blocks)
show_results()

,P@R0.7,P@R сглаж.,P@R0.7 по дням,PR-AUC,неделя с 11.04,неделя с 13.04
константа,0.079,0.079,0.080,0.079,0.082,0.079
baseline из quickstart (RF на 2 признаках),0.093,0.095,0.097,0.206,0.096,0.093
"финальная модель: CatBoost, 3 seed",0.913,0.877,0.909,0.836,0.892,0.894


## 6. Submission

In [12]:
# обучение на всём train; history для test — по событиям прошлой недели (включая test-куки более ранних дней:
# к концу окна они известны), reputation — только по меткам train
submission = pd.DataFrame({'cookie_id': X_test_full.index, 'score': final_model(X_full, y, X_test_full)})

assert list(X_test_full.columns) == list(X_full.columns)
assert len(submission) == len(test) and submission.cookie_id.is_unique
assert set(submission.cookie_id) == set(test.cookie_id)
assert submission.score.between(0, 1).all() and submission.score.notna().all()
submission.to_csv('submission.csv', index=False)
print('submission.csv:', len(submission), 'строк, уникальных score:', submission.score.nunique())

submission.csv: 4909 строк, уникальных score: 4909
